In [ ]:
# ============================================================
# 環境設定（Colab/ローカル共通）
# ============================================================
import os

# Google Earth Engineプロジェクト ID（ご自身のGEEプロジェクトIDに変更）
GEE_PROJECT = os.environ.get('GEE_PROJECT', 'your-ee-project-id')

# 出力ディレクトリ（Colabの場合は/content/drive/MyDrive/...、ローカルの場合は任意のパス）
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', '/content/drive/MyDrive/Downscaling')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'GEE_PROJECT: {GEE_PROJECT}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')


In [23]:
# ------------------------
# 1. GEE authentication
# ------------------------
import ee
ee.Authenticate()
import os
GEE_PROJECT = os.environ.get('GEE_PROJECT', 'your-ee-project')
ee.Initialize(project=GEE_PROJECT)
import time
from google.colab import files
import geopandas as gpd

In [24]:
# ------------------------
# 2. GeoJSON Upload & Grid input with ID
# ------------------------
# GeoJSON is loaded from Earth Engine as FeatureCollection
sample_points = ee.FeatureCollection("projects/ee-kurihara-yt/assets/gcm_grid_centroid") # Change as your folder link
ids = sample_points.aggregate_array('id').getInfo()

In [25]:
# ------------------------
# 3. Setting others
# ------------------------
years = list(range(1998, 2003))
#correction = ee.Image('projects/ee-kurihara-yt/assets/CagayanRB/Correction')
gcm_proj = ee.ImageCollection("NASA/GDDP-CMIP6") \
    .filter(ee.Filter.eq('model', 'ACCESS-CM2')) \
    .filter(ee.Filter.eq('scenario', 'historical')) \
    .first().select('pr').projection()

In [26]:
# ------------------------
# 4. GSMaP setting by year
# ------------------------
gsmap_list = []
for year in years:
    start = f"{year}-01-01"
    end = f"{year+1}-01-01"
    n_days = ee.Date(end).difference(ee.Date(start), 'day')

    def make_daily_img(n):
        date = ee.Date(start).advance(n, 'day')
        return ee.ImageCollection("JAXA/GPM_L3/GSMaP/v8/operational") \
            .filterDate(date, date.advance(1, 'day')) \
            .select('hourlyPrecipRateGC') \
            .sum() \
            .reproject(crs=gcm_proj.crs(), scale=gcm_proj.nominalScale()) \
            .rename('GSMaP') \
            .set('system:time_start', date.millis())

    images = ee.List.sequence(0, n_days.subtract(1)).map(make_daily_img)
    gsmap_list.append(images)

In [27]:
# ------------------------
# 5. Merge all years
# ------------------------
all_gsmap = ee.ImageCollection(gsmap_list[0])
for imgs in gsmap_list[1:]:
    all_gsmap = all_gsmap.merge(ee.ImageCollection(imgs))


In [28]:
# ------------------------
# 6. Each point in each day
# ------------------------
def extract_per_day(img):
    date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    return img.sampleRegions(collection=sample_points, scale=gcm_proj.nominalScale(), geometries=True) \
             .map(lambda f: f.set('date', date))

extracted = all_gsmap.map(extract_per_day).flatten()


In [29]:
# ------------------------
# 7. Save
# ------------------------
for id_val in ids:
    print(f"📤 Exporting ID: {id_val}")
    task = ee.batch.Export.table.toDrive(
        collection=extracted.filter(ee.Filter.eq('id', id_val)),
        description=f'his_GSMaP_id_{id_val}',
        folder='EarthEngineExport',
        fileNamePrefix=f'his_GSMaP_id_{id_val}',
        fileFormat='CSV',
        selectors=['date', 'GSMaP']
    )
    task.start()
    time.sleep(1)

📤 Exporting ID: 0
